<a href="https://colab.research.google.com/github/Amulya-Rao-mj/GENAI-lab/blob/main/rag_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu pymupdf gradio pandas

import os
import re
import json
import tempfile
import pandas as pd
import numpy as np
import torch
import fitz
import faiss
import gradio as gr

from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig
)

# ------------------------------------------------------------
# 1. CHECK HARDWARE
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "Please enable a GPU in Colab: "
        "Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. LOAD FREE PUBLIC MODELS
# ------------------------------------------------------------

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
NLI_NAME = "cross-encoder/nli-deberta-v3-small"

print("Loading Qwen answer-generation model...")

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quant_config,
    device_map="auto"
)

print("Loading evidence-evaluation model...")

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_NAME)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_NAME
).to("cuda")

nli_model.eval()

label_map = {
    int(k): str(v).lower()
    for k, v in nli_model.config.id2label.items()
}

print("NLI labels:", label_map)

if not all(
    any(name in label for label in label_map.values())
    for name in ["entailment", "neutral", "contradiction"]
):
    raise RuntimeError(
        "The NLI model does not expose the expected labels."
    )

print("Loading embedding model...")

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

print("All models loaded!")


# ------------------------------------------------------------
# 3. EXTRACT TEXT FROM PDFS AND CREATE CHUNKS
# ------------------------------------------------------------

def process_pdfs(file_paths, chunk_size=1000, overlap=200):
    chunks = []

    for file_path in file_paths:
        with fitz.open(file_path) as pdf:
            for page_num, page in enumerate(pdf, start=1):
                text = " ".join(page.get_text("text").split())

                if not text:
                    continue

                step = chunk_size - overlap

                for start in range(0, len(text), step):
                    piece = text[start:start + chunk_size]

                    if piece.strip():
                        chunks.append({
                            "source": Path(file_path).name,
                            "page": page_num,
                            "text": piece
                        })

                    if start + chunk_size >= len(text):
                        break

    if not chunks:
        raise ValueError(
            "No readable PDF text found. "
            "Scanned PDFs need OCR before uploading."
        )

    texts = [c["text"] for c in chunks]

    vectors = embedder.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)

    return chunks, index


# ------------------------------------------------------------
# 4. RETRIEVE RELEVANT PDF PASSAGES
# ------------------------------------------------------------

def retrieve(question, chunks, index, top_k=5):
    if index is None or not chunks:
        return []

    query_vector = embedder.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    k = min(top_k, len(chunks))
    scores, ids = index.search(query_vector, k)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            item = dict(chunks[int(idx)])
            item["score"] = float(score)
            results.append(item)

    return results


# ------------------------------------------------------------
# 5. GENERATE GROUNDED ANSWERS USING QWEN
# ------------------------------------------------------------

def generate_answer(question, results):
    if not results:
        return "I could not find relevant evidence in the PDFs."

    context = "\n\n".join(
        f"[Source: {r['source']}, Page: {r['page']}]\n{r['text']}"
        for r in results
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions using only the supplied PDF evidence. "
                "Cite factual claims as [filename, Page N]. "
                "Never invent facts, quotations, or page numbers. "
                "If the answer is not supported by the evidence, "
                "say that the PDFs do not provide enough information. "
                "Do not use outside knowledge."
            )
        },
        {
            "role": "user",
            "content": (
                f"PDF evidence:\n{context}\n\n"
                f"Question: {question}\n\n"
                "Answer clearly and cite the relevant pages."
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(llm.device)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.05
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


# ------------------------------------------------------------
# 6. SPLIT ANSWER INTO CLAIMS
# ------------------------------------------------------------

def split_claims(answer):
    # Remove standalone citation fragments and empty sentences.
    sentences = re.split(r"(?<=[.!?])\s+", answer.strip())

    return [
        s.strip()
        for s in sentences
        if len(s.strip().split()) >= 4
    ]


# ------------------------------------------------------------
# 7. CHECK CLAIMS AGAINST EVIDENCE USING NLI
# ------------------------------------------------------------

def check_claim(claim, passage):
    inputs = nli_tokenizer(
        passage,
        claim,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(nli_model.device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits[0]
        probs = torch.softmax(logits, dim=-1).cpu().tolist()

    scores = {
        label_map[i]: float(probs[i])
        for i in range(len(probs))
    }

    return {
        "entailment": scores.get("entailment", 0.0),
        "neutral": scores.get("neutral", 0.0),
        "contradiction": scores.get("contradiction", 0.0)
    }


def analyze_hallucinations(answer, results):
    claims = split_claims(answer)

    if not claims:
        return {
            "faithfulness": None,
            "hallucination_rate": None,
            "contradiction_rate": None,
            "claims_analyzed": 0,
            "claim_details": [],
            "note": "No assessable claims were detected."
        }

    details = []

    for claim in claims:
        comparisons = []

        for r in results:
            scores = check_claim(claim, r["text"])

            comparisons.append({
                "source": r["source"],
                "page": r["page"],
                **scores
            })

        # Best support across the retrieved passages.
        best_support = max(
            comparisons,
            key=lambda x: x["entailment"]
        )

        # Strongest contradiction across the retrieved passages.
        best_contradiction = max(
            comparisons,
            key=lambda x: x["contradiction"]
        )

        support = best_support["entailment"]
        contradiction = best_contradiction["contradiction"]

        if support >= 0.70:
            status = "SUPPORTED"
        elif contradiction >= 0.70:
            status = "CONTRADICTED"
        else:
            status = "UNVERIFIED"

        details.append({
            "claim": claim,
            "status": status,
            "support_score": round(support, 3),
            "contradiction_score": round(contradiction, 3),
            "evidence_source": best_support["source"],
            "evidence_page": best_support["page"]
        })

    total = len(details)
    supported = sum(x["status"] == "SUPPORTED" for x in details)
    contradicted = sum(x["status"] == "CONTRADICTED" for x in details)
    unverified = sum(x["status"] == "UNVERIFIED" for x in details)

    return {
        "faithfulness": round(100 * supported / total, 2),
        "hallucination_rate": round(100 * unverified / total, 2),
        "contradiction_rate": round(100 * contradicted / total, 2),
        "claims_analyzed": total,
        "supported_claims": supported,
        "unverified_claims": unverified,
        "contradicted_claims": contradicted,
        "claim_details": details,
        "note": (
            "These are NLI-based estimates against retrieved evidence. "
            "Unverified claims are not necessarily false. "
            "The thresholds should be validated against human labels."
        )
    }


# ------------------------------------------------------------
# 8. EVALUATE ANSWER RELEVANCY AND RETRIEVAL
# ------------------------------------------------------------

def evaluate_rag(question, answer, results):
    report = analyze_hallucinations(answer, results)

    vectors = embedder.encode(
        [question, answer],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    cosine = float(np.dot(vectors[0], vectors[1]))

    # Rescale cosine similarity for easier display.
    report["answer_relevancy"] = round(
        max(0.0, min(1.0, (cosine + 1.0) / 2.0)) * 100,
        2
    )

    report["retrieved_chunks"] = len(results)
    report["retrieved_sources"] = [
        {
            "source": r["source"],
            "page": r["page"],
            "similarity": round(r["score"], 3)
        }
        for r in results
    ]

    return report


# ------------------------------------------------------------
# 9. GRADIO APP: UPLOAD, ASK, EVALUATE, EXPORT
# ------------------------------------------------------------

def build_knowledge_base(files):
    if not files:
        return "Upload at least one PDF.", [], None

    try:
        file_paths = [
            f if isinstance(f, str) else f.name
            for f in files
        ]

        chunks, index = process_pdfs(file_paths)

        status = (
            f"Knowledge base ready: {len(chunks)} chunks "
            f"from {len(set(c['source'] for c in chunks))} PDF(s)."
        )

        return status, chunks, index

    except Exception as e:
        return f"PDF processing failed: {e}", [], None


def ask_question(question, chunks, index, previous_rows):
    if index is None or not chunks:
        return (
            "Upload PDFs and build the knowledge base first.",
            "",
            {},
            None,
            previous_rows or []
        )

    if not question.strip():
        return "Enter a question.", "", {}, None, previous_rows or []

    try:
        results = retrieve(question, chunks, index, top_k=5)
        answer = generate_answer(question, results)
        report = evaluate_rag(question, answer, results)

        evidence = "\n\n".join(
            f"### {r['source']} — Page {r['page']}\n"
            f"Similarity: {r['score']:.3f}\n\n{r['text']}"
            for r in results
        )

        rows = list(previous_rows or [])

        rows.append({
            "question": question,
            "answer": answer,
            "faithfulness": report.get("faithfulness"),
            "answer_relevancy": report.get("answer_relevancy"),
            "hallucination_rate": report.get("hallucination_rate"),
            "contradiction_rate": report.get("contradiction_rate"),
            "claim_details": json.dumps(
                report.get("claim_details", []),
                ensure_ascii=False
            ),
            "retrieved_sources": json.dumps(
                report.get("retrieved_sources", []),
                ensure_ascii=False
            )
        })

        report_path = os.path.join(
            tempfile.gettempdir(),
            "rag_evaluation_report.csv"
        )

        pd.DataFrame(rows).to_csv(report_path, index=False)

        return answer, evidence, report, report_path, rows

    except Exception as e:
        return (
            f"An error occurred: {e}",
            "",
            {"error": str(e)},
            None,
            previous_rows or []
        )


with gr.Blocks(title="PDF RAG and Hallucination Analysis") as demo:
    gr.Markdown(
        """
        # PDF RAG System
        Upload your PDFs, ask questions, inspect the evidence,
        and evaluate answer faithfulness and hallucination risk.
        """
    )

    chunks_state = gr.State([])
    index_state = gr.State(None)
    rows_state = gr.State([])

    pdf_upload = gr.File(
        label="Upload PDF documents",
        file_count="multiple",
        file_types=[".pdf"],
        type="filepath"
    )

    build_button = gr.Button(
        "Build Knowledge Base",
        variant="primary"
    )

    status_box = gr.Textbox(
        label="Knowledge base status",
        interactive=False
    )

    build_button.click(
        fn=build_knowledge_base,
        inputs=[pdf_upload],
        outputs=[status_box, chunks_state, index_state]
    )

    gr.Markdown("## Ask a question")

    question_box = gr.Textbox(
        label="Question",
        placeholder="What are the main findings in the PDF?"
    )

    ask_button = gr.Button(
        "Generate Answer and Evaluate",
        variant="primary"
    )

    answer_box = gr.Markdown(label="Answer")

    with gr.Accordion("Retrieved PDF evidence", open=True):
        evidence_box = gr.Markdown()

    with gr.Accordion("Evaluation and hallucination report", open=True):
        report_box = gr.JSON()

    download_box = gr.File(label="Download cumulative evaluation CSV")

    inputs = [
        question_box,
        chunks_state,
        index_state,
        rows_state
    ]

    outputs = [
        answer_box,
        evidence_box,
        report_box,
        download_box,
        rows_state
    ]

    ask_button.click(
        fn=ask_question,
        inputs=inputs,
        outputs=outputs
    )

    question_box.submit(
        fn=ask_question,
        inputs=inputs,
        outputs=outputs
    )

demo.launch(share=True, debug=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 85.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All models loaded!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d381e0e0f0772d79de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
